### This notebook calculates the imapct chains using the BD_RiesgoClimatico_IKI sql DB

**Created:** 12/09/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/26/2025 by Sophia Bakar
 
**Status:** in progress

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\

**Objective:**   

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3, matplotlib ...  

**Further documentation:**  
 
**Inputs:**   

**Outputs:** 
 
**Assumptions:** Assumes that the min and max values listed in the Indicators table of the database is correct/appropriate for normalizing the indicator values using a min/max aproach.
 
**Future work:** 
 
**Notes:** This script queries the IKI Climate Risk SQL database to generate the impact chains for the desired set of user's weights and selected scenario. Prior to calculating the impact chain, we normalize the values using a min/max approach. The resulting impact chain values will be populated into a new table "ImpactChain_Results". We use the 'Order' column from the Indicators table to determine if we should use the inverse value of an indicator. This is because for some of the indicators high values = high risk (ASC) and some of the indicators high values = low risk (DESC). When we use the inverse, we do 1- the normalized value.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import sqlite3
import os

In [2]:
# Connect to the SQLite database
# user = 'sgilson'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Externo\PACA_Peru\Indicadores\BD_RiesgoClimatico_IKI.db"
conn = sqlite3.connect(db_path)

In [3]:
# Create Impact Chain Results table 

cursor = conn.cursor()

#cursor.execute("DROP TABLE IF EXISTS ImpactChain_Results;") # uncomment to reset table/make changes to table structure

cursor.execute("""
CREATE TABLE IF NOT EXISTS ImpactChain_Results (
    IcID INTEGER,
    WaScnID INTEGER,           
    UserID INTEGER,
    COMID INTEGER,
    Peligro REAL,
    Exposicion REAL,
    VSS REAL,
    VSB REAL,
    VCA REAL,
    Vulnerabilidad REAL,
    Riesgo REAL,
    UNIQUE (WaScnID, IcID, UserID, COMID)
);
""")
conn.commit()


In [3]:
# function/sql query to get indicator data and weights for each impact chain and scenario
def get_indicator_data(wascn_id, ic_id, user_id, conn):
    """
    Returns a dataframe of indicator values and weights for one Impact Chain + one WaScnID.
    Includes Factor, TextID, WeightValue, Min, Max, and Order.
    """
    query = """
    WITH
    -- map WaScnID -> ScnID
    ScnMap AS (
        SELECT WaScnID, ScnID
        FROM WaScenarios
        WHERE WaScnID = ?
    ),

    -- WaALLOC values
    WaVals AS (
        SELECT IndID, COMID, Value, WaScnID
        FROM IndValues_WaALLOC
        WHERE WaScnID = ?
    ),

    -- Dynamic values for mapped ScnID
    DynVals AS (
        SELECT ivd.IndID, ivd.COMID, ivd.Value, sm.WaScnID
        FROM IndValues_Dyn ivd
        JOIN ScnMap sm ON ivd.ScnID = sm.ScnID
    ),

    -- Static values (applied everywhere)
    StaticVals AS (
        SELECT ivs.IndID, ivs.COMID, ivs.Value, sm.WaScnID
        FROM IndValues_Static ivs
        JOIN ScnMap sm ON 1=1
    ),

    -- Combine all values
    AllVals AS (
        SELECT * FROM WaVals
        UNION ALL
        SELECT * FROM DynVals
        UNION ALL
        SELECT * FROM StaticVals
    )

    SELECT
        av.WaScnID,
        sm.ScnID,
        ? AS UserID,
        ici.IcID,
        ici.IndID,
        av.COMID,
        av.Value,
        iw.Factor,
        iw.TextID,
        iw.WeightValue,
        ind.Min AS IndMin,
        ind.Max AS IndMax,
        ind."Order" AS IndOrder
    FROM ImpactChain_Indicators ici
    JOIN AllVals av ON ici.IndID = av.IndID
    JOIN ScnMap sm ON av.WaScnID = sm.WaScnID
    LEFT JOIN IndicatorWeights iw
        ON ici.IndID = iw.IndID AND iw.UserID = ?
    JOIN Indicators ind
        ON ici.IndID = ind.IndID
    WHERE ici.IcID = ?;
    """

    df = pd.read_sql(query, conn, params=(wascn_id, wascn_id, user_id, user_id, ic_id))
    return df


In [4]:
# helper function to perform QC checks on indicator ranges for min/max values
def qc_check_indicator_ranges(df):
    """
    Print observed vs table min/max for QC.
    """
    print("\nQC CHECK: Indicator ranges")
    for ind_id, g in df.groupby("IndID"):
        obs_min = g["Value"].min()
        obs_max = g["Value"].max()
        tbl_min = g["IndMin"].iloc[0]
        tbl_max = g["IndMax"].iloc[0]

        print(
            f"IndID {ind_id}: "
            f"Observed min/max = ({obs_min:.3f}, {obs_max:.3f}) | "
            f"Table min/max = ({tbl_min:.3f}, {tbl_max:.3f})"
        )


In [5]:
# function to perform the min/max normalization
def min_max_normalize(df):
    """
    Apply min–max scaling using Indicators.Min and Indicators.Max.

    Then apply directionality:
    - ASC  => keep normalized values
    - DESC => invert normalized values (1 - norm)
    """
    df = df.copy()

    def scale(row):
        if pd.isna(row["Value"]):
            return np.nan

        denom = row["IndMax"] - row["IndMin"]
        if denom == 0:
            return 0.0

        norm = (row["Value"] - row["IndMin"]) / denom
        norm = np.clip(norm, 0, 1)

        # Apply ASC/DESC inversion
        order_val = str(row.get("IndOrder", "ASC")).strip().upper()
        if order_val == "DESC":
            norm = 1 - norm

        return norm

    df["ValueNorm"] = df.apply(scale, axis=1)
    return df

In [6]:
# compute weighted index for Peligro and Exposición Indicators
def compute_weighted_index(df, factor_name):
    """
    Compute weighted average for factor 'P' or 'E'.

    Rules:
    - Value == 0  → weight included
    - Value == NaN → weight excluded
    """
    df_factor = df[df["Factor"] == factor_name].copy()
    if df_factor.empty:
        return 0.0

    vals = pd.to_numeric(df_factor["Value"], errors="coerce")
    ws   = pd.to_numeric(df_factor["WeightValue"], errors="coerce")

    # keep only rows with non-NaN values
    mask = ~vals.isna()
    if not mask.any():
        return 0.0

    vals = vals[mask]
    ws   = ws[mask]

    denom = ws.sum()
    if denom == 0:
        return 0.0

    return float((vals * ws).sum() / denom)

In [7]:
# compute weighted index for Vulnerability Indicators
def compute_vulnerability(df):
    """
    Compute Vulnerability:

    Vulnerabilidad =
      (VSB * W_vsb + VSS * W_vss + VCA * W_vca)
      / (W_vsb + W_vss + W_vca)

    Rules:
    - Use normalized values (ValueNorm)
    - Indicator value == NaN → weight excluded
    """
    df_v = df[df["Factor"] == "V"].copy()
    
    if df_v.empty:
        return 0.0, 0.0, 0.0, 0.0

    categories = ["VSB", "VSS", "VCA"]
    results = {}

    for cat in categories:
        sub = df_v[df_v["TextID"].fillna("").str.strip().str.upper().str.startswith(cat)].copy()
        if sub.empty:
            results[cat] = {"avg": 0.0, "w_sum": 0.0}
            continue

        vals = pd.to_numeric(sub["ValueNorm"], errors="coerce")  # <-- USE NORMALIZED
        ws   = pd.to_numeric(sub["WeightValue"], errors="coerce")

        mask = ~vals.isna()
        if not mask.any():
            results[cat] = {"avg": 0.0, "w_sum": 0.0}
            continue

        vals = vals[mask]
        ws   = ws[mask]

        w_sum = ws.sum()
        avg = (vals * ws).sum() / w_sum if w_sum != 0 else 0.0

        results[cat] = {"avg": float(avg), "w_sum": float(w_sum)}

    # unpack
    VSB, w_vsb = results["VSB"]["avg"], results["VSB"]["w_sum"]
    VSS, w_vss = results["VSS"]["avg"], results["VSS"]["w_sum"]
    VCA, w_vca = results["VCA"]["avg"], results["VCA"]["w_sum"]

    denom = w_vsb + w_vss + w_vca
    if denom == 0:
        return 0.0, VSS, VSB, VCA

    vulnerabilidad = (VSB * w_vsb + VSS * w_vss + VCA * w_vca) / denom

    return float(vulnerabilidad), VSS, VSB, VCA

In [8]:
def get_factor_weights(ic_id, user_id):
    """
    Returns a dict with keys 'P','V','E' mapping to weight values.
    Expects FactorWeights to have Factor column with values 'P','V','E'.
    """
    q = """
        SELECT Factor, WeightValue
        FROM FactorWeights
        WHERE IcID = ? AND UserID = ?;
    """
    df = pd.read_sql(q, conn, params=(ic_id, user_id))

    # No user-defined weights at all → equal weights
    if df.empty:
        return {"P": 1/3, "V": 1/3, "E": 1/3}

    # Partial user-defined weights → fill missing with 1/3
    weights = {
        row["Factor"]: float(row["WeightValue"])
        for _, row in df.iterrows()
    }

    return {
        "P": weights.get("P", 1/3),
        "V": weights.get("V", 1/3),
        "E": weights.get("E", 1/3)
    }


In [9]:
# replace values based on User, Scenario, COMID, and Impact Chain
def insert_results(
    wascn_id, ic_id, user_id, comid,
    p, e, vss, vsb, vca, vulnerabilidad, r
):
    q = """
        INSERT OR REPLACE INTO ImpactChain_Results
        (WaScnID, IcID, UserID, COMID,
         Peligro, Exposicion, VSS, VSB, VCA,
         Vulnerabilidad, Riesgo)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """
    conn.execute(q, (
        wascn_id, ic_id, user_id, comid,
        p, e, vss, vsb, vca, vulnerabilidad, r
    ))
    conn.commit()

In [10]:
# function to process an impact chain for a given scenario and user; 
def process_ic_scenario(wascn_id, ic_id, user_id, run_qc, conn):
    # Get indicator data (includes Min/Max and Order)
    df = get_indicator_data(wascn_id, ic_id, user_id, conn)

    expected_inds = pd.read_sql(
        """
        SELECT IndID
        FROM ImpactChain_Indicators
        WHERE IcID = ?
        """,
        conn,
        params=(ic_id,)
    )["IndID"].unique()

    if df.empty:
        print(f"No data for IcID={ic_id}, WaScnID={wascn_id}, UserID={user_id}")
        return

    present_inds = df["IndID"].unique()
    missing_inds = sorted(set(expected_inds) - set(present_inds))

    if missing_inds:
        print(
            f"WARNING: Missing indicator values for "
            f"IcID={ic_id}, WaScnID={wascn_id}, UserID={user_id} → "
            f"Missing IndIDs: {missing_inds}"
        )

    # Optional QC check
    if run_qc:
        qc_check_indicator_ranges(df)

    # Apply min-max normalization (includes ASC/DESC inversion)
    df = min_max_normalize(df)

    # Use normalized values for all calculations
    df["Value"] = df["ValueNorm"]

    # Get user-specific factor weights
    factor_weights = get_factor_weights(ic_id, user_id)

    # Group by COMID
    for comid, g in df.groupby("COMID"):
        # Compute Peligro and Exposición (weighted average)
        p = compute_weighted_index(g, "P")
        e = compute_weighted_index(g, "E")

        # Compute Vulnerabilidad (weighted avg of VSB, VSS, VCA)
        v, vss, vsb, vca = compute_vulnerability(g)

        # Aggregate Risk using factor weights
        fw = factor_weights
        denom = fw["P"] + fw["V"] + fw["E"]
        r = (p * fw["P"] + v * fw["V"] + e * fw["E"]) / denom if denom != 0 else np.mean([p, v, e])

        # Insert results into DB
        insert_results(
            wascn_id, ic_id, user_id, comid,
            p, e, vss, vsb, vca, v, r
        )


In [11]:
impact_chains = pd.read_sql(
        "SELECT DISTINCT IcID FROM ImpactChain_Indicators;",
        conn
    )["IcID"].tolist()

In [13]:
# to run all impact chains, all users, and all scenarios (UPDATE)
if __name__ == "__main__":
    user_ids = [1]                 # select users, we can set this up later to loop over all users in the database
    scenarios = [1, 3, 7]          # update based on scenarios, we can set this up later to loop over all scenarios in the WaScenarios table
    run_qc = False                 # set run_qc = True to perform QC check on min and max, otherwise set False

    impact_chains = pd.read_sql(
        "SELECT DISTINCT IcID FROM ImpactChain_Indicators;",
        conn
    )["IcID"].tolist()

    for wascn_id in scenarios:
        for ic in impact_chains:
            for user_id in user_ids:
                print(f"Processing: WaScnID={wascn_id}, IcID={ic}, UserID={user_id}")
                process_ic_scenario(
                    wascn_id=wascn_id,
                    ic_id=ic,
                    user_id=user_id,
                    run_qc=run_qc,
                    conn=conn
                )

    conn.close()

Processing: WaScnID=1, IcID=1, UserID=1
Processing: WaScnID=1, IcID=2, UserID=1
Processing: WaScnID=1, IcID=3, UserID=1
Processing: WaScnID=1, IcID=4, UserID=1
Processing: WaScnID=1, IcID=5, UserID=1
Processing: WaScnID=1, IcID=6, UserID=1
Processing: WaScnID=1, IcID=7, UserID=1
Processing: WaScnID=1, IcID=8, UserID=1
Processing: WaScnID=1, IcID=9, UserID=1
Processing: WaScnID=1, IcID=10, UserID=1
Processing: WaScnID=1, IcID=11, UserID=1
Processing: WaScnID=1, IcID=12, UserID=1
Processing: WaScnID=1, IcID=13, UserID=1
Processing: WaScnID=1, IcID=14, UserID=1
Processing: WaScnID=1, IcID=15, UserID=1
Processing: WaScnID=1, IcID=16, UserID=1
Processing: WaScnID=1, IcID=17, UserID=1
Processing: WaScnID=1, IcID=18, UserID=1
Processing: WaScnID=1, IcID=19, UserID=1
Processing: WaScnID=1, IcID=20, UserID=1
Processing: WaScnID=3, IcID=1, UserID=1
Processing: WaScnID=3, IcID=2, UserID=1
Processing: WaScnID=3, IcID=3, UserID=1
Processing: WaScnID=3, IcID=4, UserID=1
Processing: WaScnID=3, IcID=5

In [12]:
if __name__ == "__main__":
    user_ids = [1]                  # select users
    scenarios = [1, 3]              # selected scenarios
    run_qc = False                   # QC check flag

    # Define subset of impact chains you want to process
    impact_chains_subset = [1,3,5,6,8,18,20]  # <-- replace with your desired IcID values

    for wascn_id in scenarios:
        for ic in impact_chains_subset:
            for user_id in user_ids:
                print(f"Processing: WaScnID={wascn_id}, IcID={ic}, UserID={user_id}")
                process_ic_scenario(
                    wascn_id=wascn_id,
                    ic_id=ic,
                    user_id=user_id,
                    run_qc=run_qc,
                    conn=conn
                )

    conn.close()

Processing: WaScnID=1, IcID=1, UserID=1
Processing: WaScnID=1, IcID=3, UserID=1
Processing: WaScnID=1, IcID=5, UserID=1
Processing: WaScnID=1, IcID=6, UserID=1
Processing: WaScnID=1, IcID=8, UserID=1
Processing: WaScnID=1, IcID=18, UserID=1
Processing: WaScnID=1, IcID=20, UserID=1
Processing: WaScnID=3, IcID=1, UserID=1
Processing: WaScnID=3, IcID=3, UserID=1
Processing: WaScnID=3, IcID=5, UserID=1
Processing: WaScnID=3, IcID=6, UserID=1
Processing: WaScnID=3, IcID=8, UserID=1
Processing: WaScnID=3, IcID=18, UserID=1
Processing: WaScnID=3, IcID=20, UserID=1
